# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Rydwan da Silva]
**Student ID:** [22492028]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

from dotenv import load_dotenv
from openai import OpenAI

# --- Local (with a .env file) ---

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):


client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",  # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"  # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:


def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500,
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage


# TODO: Call it once with a simple question and print the answer.
response, usage = ask_llm("What is the capital of France?")
print(response)
# TODO: Print response.usage as well — how many tokens did your call consume?
print("Token usage:", usage)

The capital of France is Paris.
Token usage: CompletionUsage(completion_tokens=8, prompt_tokens=48, total_tokens=56, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.042240143, prompt_time=0.001436764, completion_time=0.011294797, total_time=0.012731561)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
*1. The system refers to developer-written instructions that determine how the model should behave throughout the conversation and the user refers to the actual input from the human interacting with the model at runtime. An example of something that goes in each:
system prompt: You are a weather assistant with a lot of experience in the field.
user prompt: What is the weather forecast in Accra?
*2. A token is a chunk of text roughly 3–4 characters. API providers bill per token rather than per request because token count directly measures how much compute the model consumed — more tokens means more attention operations and more GPU time.

### Part 1.2 — Temperature: the randomness dial

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
for temp in [0.0, 1.2]:
    print(f"\nTemperature: {temp}")
    for i in range(5):
        response, usage = ask_llm(
            "Suggest a name for a savings product for market traders in Accra.",
            temperature=temp,
        )
        print(f"Answer {i + 1}: {response} (Tokens used: {usage})")



Temperature: 0.0
Answer 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders.
5. **Sika Kurom**: "Sika" means "money" in Ghanaian, and "Kurom" means "box" or "container", so this name suggests a safe and secure place to store savings.
6. **Traders' Fund**: This name is straightforward and emphasizes the idea of a collective fund for market traders.
7. **Adanfo Save**: "Adanfo" is a Ghanaian word for "friends" or "partners", so this name suggests a sense of community and cooperation.

Choose the one

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** With the small temperature of 0.0, the answers were almost the same for all five attempts, while at 1.2, the answers varied a lot each time. For the system I'm about to build it is preferrable to have consistent answers so a smaller temperature would be ideal. 

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
    "L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",
    "L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",
    "L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",
    "L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",
    "L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
    "L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20,
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15,
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12,
    },
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this loan application:\n\n{letter_text}"

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_PROMPT_V2 = """You are an assistant to a microfinance loan officer. Your task is to summarize loan applications in a factual and neutral manner, without inventing any details. Please provide a concise summary in 3-4 sentences.
Summarize this loan application:{letter_text}"""

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
response_v1_02, usage_v1_02 = ask_llm(
    SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"]), temperature=0.0
)
response_v2_02, usage_v2_02 = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"]), temperature=0.0
)
response_v1_06, usage_v1_06 = ask_llm(
    SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"]), temperature=0.0
)
response_v2_06, usage_v2_06 = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"]), temperature=0.0
)
print("\nSummary for L002:")
print("V1 Summary:", response_v1_02)
print("V1 Token usage:", usage_v1_02)
print("\nV2 Summary:", response_v2_02)
print("V2 Token usage:", usage_v2_02)

print("\nSummary for L006:")
print("V1 Summary:", response_v1_06)
print("V1 Token usage:", usage_v1_06)
print("\nV2 Summary:", response_v2_06)
print("V2 Token usage:", usage_v2_06)


Summary for L002:
V1 Summary: Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his vehicle's engine and pay off personal debts. His business has been slow, but he expects it to improve after the festive season. He doesn't have collateral to offer and is relying on his future earnings to repay the loan. He is seeking urgent assistance.
V1 Token usage: CompletionUsage(completion_tokens=85, prompt_tokens=135, total_tokens=220, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041552736, prompt_time=0.006788708, completion_time=0.231019303, total_time=0.237808011)

V2 Summary: Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not currently have collateral to offer, but is s

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** 
*1. While I don't notice any big difference between both outputs I can say that the output for V2 was more factual. For the L002, the V2 summary mentions trotro while v1 stays generic by saying vehicle. 
*2. The "no invented details" is an essential instruction in this application because it reminds the model not to add any non-existent information in the output. This failure is called hallucination in LLM literature and refers to the fact that LLMs make up false information confidently just to provide an answer.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [6]:
import json
import pandas as pd
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0


EXTRACT_PROMPT = """You are an assistant to a microfinance loan officer. Your task is to extract specific information from loan applications and return it in a structured JSON format. Please provide a JSON object with the following keys:
- applicant_name (string or null)
- amount_ghs (number or null)
- purpose (string or null)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean or null)
- repayment_months (number or null)
If a field is not stated in the letter, use null. Do not guess.
Here is an example letter and the expected JSON output to guide you:
Letter: 
Dear Loan Officer,

My name is Abena Mensah and I am writing to apply for a loan of GHS 3,500 
to purchase additional sewing machines and fabric stock for my tailoring 
business located in Kumasi. I have been operating this business for four 
years and currently earn a monthly profit of approximately GHS 800. 

My neighbour, Mr. Kofi Darko, has agreed to serve as my guarantor for this 
loan. I would like to repay the loan over 12 months.

Thank you for your consideration.

Yours sincerely,
Abena Mensah

expected JSON: 
    {{
        "applicant_name": "Abena Mensah",
        "amount_ghs": 3500,
        "purpose": "to purchase additional sewing machines and fabric stock for my tailoring business located in Kumasi",
        "monthly_profit_ghs": 800,
        "has_collateral_or_guarantor": true,
        "repayment_months": 12
    }}
Now, extract the information from the following letter:{letter_text}
Return ONLY the JSON object, without any additional text or explanation."""


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text, temperature=0.0):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)
    response, _usage = ask_llm(prompt, temperature=temperature)

    # Strip any ```json fences if present
    cleaned_response = response.strip()
    if cleaned_response.startswith("```json"):
        cleaned_response = cleaned_response[len("```json")]  # Remove the opening and closing fences
    elif cleaned_response.startswith("```"):
        cleaned_response = cleaned_response[len("```"):]  # Remove the opening fence
    if cleaned_response.endswith("```"):
        cleaned_response = cleaned_response[:-3]  # Remove the closing fence
    cleaned_response = cleaned_response.strip()
      # Remove any leading/trailing whitespace

    try:
        data = json.loads(cleaned_response)
        return data
    except json.JSONDecodeError as e:
        print(f"Warning: Failed to parse JSON for letter. Error: {e}")
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

results = []

for letter_id, letter_text in LETTERS.items():
    extracted_data = extract_fields(letter_text)

    if extracted_data is not None:
        extracted_data["letter_id"] = letter_id
        results.append(extracted_data)
    else:
        print(f"Warning: Extraction failed for letter {letter_id}.")
df_extracted = pd.DataFrame(results)
df_extracted


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,to buy a deep freezer and expand into frozen f...,900.0,True,20.0,L001
1,Kwame Boateng,25000,to repair my trotro engine and settle some per...,NaN,False,NaN,L002
2,Efua Darko,15000,to purchase two industrial sewing machines and...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0,L004
4,NaN,30000,to buy a bulk order of yarn directly from the ...,NaN,True,16.0,L005
5,Kofi,50000,"to start a car washing business, a provision s...",NaN,False,12.0,L006


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** 
*1. Because we don't want the model to simply memorize the expected output but to be able to generalize to other inputs too
*2. This made the model avoid hallucination. Without it, the model was assigning whatever type it thought was appropriate and when the actual input did not contain any value for the key the json load would crash or the model would just add a random value to fill the gap.
*3. Keeping the temperature at 0 makes the result consistent everytime we call the model. This is the right choice because we wouldn't want the answers to differ in such a task. For creative tasks on the other hand, since we would want to diversify the output we would want the temperature to be higher so that the model also outputs results with different probabilities instead of always the one with the highest. 

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [7]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT = """You are an assistant to a microfinance loan officer. Your task is to provide a brief analysis of a loan application based on the letter and the extracted information. Please provide the following in your response:
1. Strengths (bullet points, grounded in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents", "flag for senior review") — NOT "approve" or "reject".
Please ensure that your analysis is factual, neutral, and based solely on the information provided in the letter and the extracted data. Do not invent any details or make assumptions beyond what is explicitly stated. 
Final decisions are made by human officers. Here is the letter and the extracted information in JSON format:
Letter:{letter_text}
Extracted JSON:{extracted_json}
Return your analysis in a clear, structured format with numbered sections as specified above."""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
for letter in LETTERS.keys():
    letter_text = LETTERS[letter]
    extracted_json = df_extracted[df_extracted["letter_id"] == letter].to_dict(orient="records")[0]
    prompt = BRIEF_PROMPT.format(letter_text=letter_text, extracted_json=json.dumps(extracted_json))
    brief_response, _usage = ask_llm(prompt, temperature=0.0)
    if letter in ["L001", "L002", "L003", "L006"]:
        print(f"\nBrief for {letter}:\n{brief_response}\n")


Brief for L001:
## 1. Strengths
* The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market.
* She has a stable monthly profit of GHS 900, which indicates a consistent income stream.
* Akosua has demonstrated a savings habit through the susu scheme, accumulating GHS 2,500 over two years without missing any contributions.
* She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of security for the loan.
* The applicant has a clear plan for using the loan, which is to buy a deep freezer and expand into frozen foods, and has proposed a repayment plan of GHS 450 monthly over 20 months.

## 2. Risks / Red Flags
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit of GHS 900, which might pose a risk if the expansion into frozen foods does not generate sufficient additional income.
* There is no detailed information provided about the sister's financial stabil

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** 

*1 Yes, the model successfuly identified the right strengths and red flags for letter L003 and letter L006. We also clearly see that letter L003 has more strenght bullet points than L006 and less red flags than L003

*2. There is a critical issue with leting the model decide by itself whether an application should be approved or not. The model can make mistakes and is can create real serious problems for the officers. By letting it provide information to officers we ensure that the system is merely simplifying the task for the offecers which in turn have full control of the final decisions.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 9cf6b95bba65932b66431d08e48984732e3a8b23

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [8]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

# Get the letter IDs from your GOLD dictionary (e.g., ['L001', 'L003', 'L006'])
letter_ids = list(GOLD.keys())

# Ensure df_extracted has letter_id as its index for easy row lookup
df_indexed = df_extracted.set_index('letter_id')

comparison_rows = []

for field in fields:
    row = {}
    matches = []
    
    for lid in letter_ids:
        ext_val = df_indexed.loc[lid, field]
        gold_val = GOLD[lid][field]
        
        # Apply matching rules: case-insensitive/stripped for strings, exact for others
        if isinstance(gold_val, str) and isinstance(ext_val, str):
            match = ext_val.strip().lower() == gold_val.strip().lower()
        else:
            match = (ext_val == gold_val)
            
        row[lid] = match
        matches.append(match)
        
    # Calculate per-field accuracy across the three letters (proportion of True values)
    row['accuracy'] = sum(matches) / len(matches)
    comparison_rows.append(row)

# Create the final comparison DataFrame
df_comparison = pd.DataFrame(comparison_rows, index=fields)

# Display the table
display(df_comparison)

,L001,L003,L006,accuracy
applicant_name,True,True,True,1.000000
amount_ghs,True,True,True,1.000000
purpose,False,False,False,0.000000
monthly_profit_ghs,True,True,False,0.666667
has_collateral_or_guarantor,True,True,True,1.000000
repayment_months,True,True,True,1.000000


### Part 4.2 — Reliability: is the system consistent?

In [9]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
letter_l004 = LETTERS["L004"]
temperatures = [0.0, 1.0]

for temp in temperatures:
    valid_json_count = 0
    serialized_runs = []
    
    for i in range(5):
        result = extract_fields(letter_l004, temperature=temp)
        
        if result is not None:
            valid_json_count += 1
            # Serialize with sorted keys to ensure consistent key ordering for comparison
            serialized_runs.append(json.dumps(result, sort_keys=True))
        else:
            serialized_runs.append(None)
            
    # Count unique string representations among successful runs
    successful_runs = [r for r in serialized_runs if r is not None]
    unique_outputs = len(set(successful_runs))
    
    print(f"--- Temperature {temp} ---")
    print(f"Valid JSON produced: {valid_json_count} / 5 runs")
    print(f"Unique output variations: {unique_outputs} (out of {valid_json_count} valid runs)")
    print(f"Identical values across runs: {'Yes' if unique_outputs == 1 and valid_json_count == 5 else 'No'}\n")

--- Temperature 0.0 ---
Valid JSON produced: 5 / 5 runs
Unique output variations: 1 (out of 5 valid runs)
Identical values across runs: Yes

--- Temperature 1.0 ---
Valid JSON produced: 5 / 5 runs
Unique output variations: 1 (out of 5 valid runs)
Identical values across runs: Yes



### Part 4.3 — Hallucination probing

In [10]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# Test 1 — Asking about a detail NOT in the letter (Credit Score)
test_letter = LETTERS["L001"] # Or any standard letter

summary_prompt = f"""Based on the following loan application letter, answer the question: What is the applicant's credit score?
If the information is not present in the text, explicitly state that it is not provided. Do not guess.

Letter:
{test_letter}"""

response_test1, _ = ask_llm(summary_prompt, temperature=0.0)
print("--- Test 1 Output ---")
print(response_test1)

# Test 2 — Feed extractor an irrelevant text (Weather report)
weather_text = (
    "Today in Accra, the weather is partly cloudy with a high of 30°C "
    "and a low of 24°C. Expect light breezes from the south-west and "
    "a 10% chance of isolated afternoon showers."
)

result_test2 = extract_fields(weather_text)
print("--- Test 2 Output ---")
print(result_test2)

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
output_test1_verbatim = response_test1
test1_result = "PASS" if "not" in output_test1_verbatim.lower() else "FAIL"
output_test2_verbatim = result_test2
test2_result = "PASS" if all(value is None for value in output_test2_verbatim.values()) else "FAIL"
print(f"Test 1 Result: {test1_result}")
print(f"Test 2 Result: {test2_result}")

--- Test 1 Output ---
The applicant's credit score is not provided in the text.
--- Test 2 Output ---
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}
Test 1 Result: PASS
Test 2 Result: PASS


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?* 
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** 
*1. The purpose field was harder to find probably because most applicant do not summit it as part of their letter
*2. The readability experiement showed that a change in the temperature did not affect the output of the model
*3. No, the system did not hallucinate under probing

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** 
*1. If the bank fully automated decisions with my system, the primary victims of systemic unfairness would be marginalized, informal-sector entrepreneurs who possess strong business acumen but lack formal English literacy or professional writing skills
*2. Regulated financial institutions have a strict legal duty to protect customer data. Transmitting customer financial profiles across international borders requires ensuring that the receiving jurisdiction or vendor provides an adequate level of data protection. Before deploying the system I need to ensure the API provider explicitly guarantees that customer data will not be used to train public models, check whether data is stored securely in compliance with local regulations or if zero-retention policies are enforced, and confirm robust end-to-end encryption (TLS in transit and AES-256 at rest) and strict access control mechanisms. 
*3. TWO CHECKS:
- The AI must strictly function as a decision-support tool rather than a decision-maker. While the LLM can pre-populate fields and draft preliminary risk summaries, every final approval, and 100% of all rejections or flagrals, must be reviewed and signed off by a human loan officer. Furthermore, any automated rejection must trigger a transparent review path where applicants can appeal or provide verbal clarifications.
- Implement comprehensive audit trails (similar to core banking audit logs) that capture the exact prompt sent, the raw LLM output, the extracted structured data, and the final decision made by the loan officer. This ensures complete transparency, accountability, and traceability for internal compliance audits and regulatory inspections by the Bank of Ghana.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** 
**1. Prompting as Engineering**
Iterating on a prompt is similar to hyperparameter tuning because both are iterative processes designed to optimize model performance without modifying underlying model weights. However, hyperparameter tuning adjusts precise quantitative knobs (like learning rate or batch size) through mathematical search, whereas prompt engineering guides model behavior using natural language instructions, and few-shot context.

**2. Trust**
No, I would not trust this system to run unattended in a live lending environment. What mostly influenced my answer was the model's vulnerability to formatting edge cases and syntax sensitivity (such as returning single-quoted dictionaries or hallucinating structure on non-standard inputs), which demonstrates that minor input variations can cause parsing failures or data loss if there is no human to supervise.

**3. Cost and Scale**
Assuming ~600 input tokens (prompt + letter text) and ~150 output tokens per run, 1,000 applications per month require approximately **750,000 total tokens** monthly. Because this volume is lightweight and costs under $1 per month on most commercial APIs, provider choice should be dictated by data privacy guarantees, zero-data-retention SLAs, and regulatory compliance rather than raw token pricing.

**4. Looking Back at the Course**
Calling a foundation model API beats training a custom model for this task because LLMs offer powerful zero-shot natural language understanding out-of-the-box, extracting structured entities from varied, unstructured human text without needing thousands of hand-labeled training samples or GPU infrastructure. Conversely, training or fine-tuning a custom model is necessary when strict financial regulations or air-gapped security policies prohibit transmitting customer data to third-party APIs, or when operating at massive scale where on-premise inference becomes cheaper.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.